In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        os.path.join(dirname, filename)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
pip install segmentation-models-pytorch

In [ ]:
import os
import numpy as np
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split
from pycocotools.coco import COCO
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

In [ ]:
TRAIN_JSON = '/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json'   # fix to actual filename
TRAIN_IMG_DIR = '/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train/train_images'                    # fix to actual image folder
IMAGE_SIZE = 512
BATCH_SIZE = 8
NUM_EPOCHS = 30
LR = 1e-4
VAL_SPLIT = 0.15
SEED = 42
PATIENCE = 6   # early stopping

torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
coco = COCO(TRAIN_JSON)
image_ids = list(coco.imgs.keys())
train_ids, val_ids = train_test_split(image_ids, test_size=VAL_SPLIT, random_state=SEED)
print(f"Train: {len(train_ids)} | Val: {len(val_ids)}")

In [ ]:
print(type(image_ids[0]), image_ids[:5])
print(type(list(coco.imgs.keys())[0]), list(coco.imgs.keys())[:5])

In [ ]:
train_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, border_mode=cv2.BORDER_CONSTANT, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.GaussNoise(var_limit=(5.0, 20.0), p=0.3),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2(),
])

In [ ]:
class FilamentDataset(Dataset):
    def __init__(self, coco, image_ids, img_dir, transform):
        self.coco = coco
        self.image_ids = image_ids
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        img_info = self.coco.loadImgs([image_id])[0]              # <-- fixed
        img_path = os.path.join(self.img_dir, img_info['file_name'])

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        img = np.expand_dims(img, axis=-1)

        ann_ids = self.coco.getAnnIds(imgIds=[image_id])          # <-- fixed
        anns = self.coco.loadAnns(ann_ids)
        masks = [self.coco.annToMask(ann) for ann in anns]
        mask = np.any(masks, axis=0).astype('float32') if masks else np.zeros(img.shape[:2], dtype='float32')

        augmented = self.transform(image=img, mask=mask)
        img_tensor = augmented['image']
        mask_tensor = augmented['mask'].unsqueeze(0).float()

        return img_tensor, mask_tensor

In [ ]:
train_dataset = FilamentDataset(coco, train_ids, TRAIN_IMG_DIR, train_transform)
val_dataset = FilamentDataset(coco, val_ids, TRAIN_IMG_DIR, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',   # pretrained encoder — this is your transfer learning
    in_channels=1,
    classes=1,
    activation=None,              # raw logits, sigmoid applied in loss/metric
).to(DEVICE)

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits).view(-1)
        targets = targets.view(-1)
        intersection = (probs * targets).sum()
        dice = (2. * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1 - dice

class BCEDiceLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

    def forward(self, logits, targets):
        return 0.5 * self.bce(logits, targets) + 0.5 * self.dice(logits, targets)

def dice_score(logits, targets, threshold=0.5, smooth=1e-6):
    preds = (torch.sigmoid(logits) > threshold).float().view(-1)
    targets = targets.view(-1)
    intersection = (preds * targets).sum()
    return (2. * intersection + smooth) / (preds.sum() + targets.sum() + smooth)

criterion = BCEDiceLoss()
optimizer = Adam(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits).view(-1)
        targets = targets.view(-1)
        intersection = (probs * targets).sum()
        dice = (2. * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1 - dice

class BCEDiceLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

    def forward(self, logits, targets):
        return 0.5 * self.bce(logits, targets) + 0.5 * self.dice(logits, targets)

def dice_score(logits, targets, threshold=0.5, smooth=1e-6):
    preds = (torch.sigmoid(logits) > threshold).float().view(-1)
    targets = targets.view(-1)
    intersection = (preds * targets).sum()
    return (2. * intersection + smooth) / (preds.sum() + targets.sum() + smooth)

criterion = BCEDiceLoss()
optimizer = Adam(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

In [ ]:
best_val_dice = 0.0
epochs_no_improve = 0
history = {'train_loss': [], 'val_loss': [], 'val_dice': []}

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
    train_loss /= len(train_dataset)

    model.eval()
    val_loss, val_dice = 0.0, 0.0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            logits = model(imgs)
            loss = criterion(logits, masks)
            val_loss += loss.item() * imgs.size(0)
            val_dice += dice_score(logits, masks).item() * imgs.size(0)
    val_loss /= len(val_dataset)
    val_dice /= len(val_dataset)

    scheduler.step()
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_dice'].append(val_dice)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f}")

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"  -> New best model saved (Dice: {best_val_dice:.4f})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs)")
            break

print(f"\nBest validation Dice: {best_val_dice:.4f}")

In [ ]:
total_target_pixels = 0
total_pred_pixels = 0

model.eval()
with torch.no_grad():
    for imgs, masks in val_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        logits = model(imgs)
        preds = (torch.sigmoid(logits) > 0.5).float()

        total_target_pixels += masks.sum().item()
        total_pred_pixels += preds.sum().item()

print(f"Total ground-truth filament pixels in val set: {total_target_pixels}")
print(f"Total predicted filament pixels in val set: {total_pred_pixels}")

In [ ]:
# check just the raw dataset, no model involved
mask_sums = [train_dataset[i][1].sum().item() for i in range(20)]
print(mask_sums)

In [ ]:
sample_id = train_ids[0]
print("Sample image_id:", sample_id)

img_info = coco.loadImgs([sample_id])[0]     # <-- wrapped in [ ]
print("Image info:", img_info)

ann_ids = coco.getAnnIds(imgIds=[sample_id])  # <-- wrapped in [ ]
print(f"Number of annotation IDs found: {len(ann_ids)}")

anns = coco.loadAnns(ann_ids)
if anns:
    print("First annotation keys:", list(anns[0].keys()))
    print("First annotation:", anns[0])
    test_mask = coco.annToMask(anns[0])
    print("Mask shape:", test_mask.shape, "| Mask sum:", test_mask.sum())
else:
    print("No annotations found for this image_id!")

In [ ]:
model.load_state_dict(torch.load('best_model.pth', map_location=DEVICE))
model.eval()
print("Best model loaded.")

In [ ]:
from scipy import ndimage
from pycocotools import mask as mask_utils

def split_into_filaments(binary_mask, min_size=20):
    labeled_mask, num_features = ndimage.label(binary_mask)
    valid_ids = [fid for fid in range(1, num_features + 1) if np.sum(labeled_mask == fid) >= min_size]
    return labeled_mask, valid_ids

def mask_to_rle(single_mask):
    single_mask = np.asfortranarray(single_mask.astype('uint8'))
    rle = mask_utils.encode(single_mask)
    return rle['counts'].decode('utf-8')

In [ ]:
TEST_IMG_DIR = '/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images'  # confirm this matches your actual test folder name

test_image_paths = sorted(
    os.path.join(TEST_IMG_DIR, f) for f in os.listdir(TEST_IMG_DIR)
)
print(f"Test images: {len(test_image_paths)}")

def predict_full_res_mask(model, image_path, train_size=IMAGE_SIZE, threshold=0.5):
    img_full = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    orig_h, orig_w = img_full.shape

    img_resized = cv2.resize(img_full, (train_size, train_size)).astype('float32') / 255.0
    img_resized = (img_resized - 0.5) / 0.5  # match Normalize(mean=0.5, std=0.5) from training
    img_tensor = torch.from_numpy(img_resized).unsqueeze(0).unsqueeze(0).float().to(DEVICE)

    with torch.no_grad():
        logits = model(img_tensor)
        probs = torch.sigmoid(logits).squeeze().cpu().numpy()

    probs_full = cv2.resize(probs, (orig_w, orig_h))
    binary_mask = (probs_full > threshold).astype('uint8')
    return binary_mask

In [ ]:
all_rows = []
for path in test_image_paths:
    image_id = os.path.splitext(os.path.basename(path))[0]
    binary_mask = predict_full_res_mask(model, path)
    labeled_mask, filament_ids = split_into_filaments(binary_mask)

    for i, fid in enumerate(filament_ids, start=1):
        single_mask = (labeled_mask == fid).astype('uint8')
        rle_string = mask_to_rle(single_mask)
        all_rows.append({'filament_id': f'{image_id}_{i}', 'segmentation_rle': rle_string})

submission_df = pd.DataFrame(all_rows)
submission_df.to_csv('submission.csv', index=False)
print(f"Saved {len(submission_df)} filament rows to submission.csv")
submission_df.head()